# P4 Lab · Transformer의 구조와 활용 (1·2절 보강판)

이 실습의 목표는 **모델을 훈련하는 것이 아니라 사전훈련된 모델을 사용하여 Transformer의 작동 원리를 관찰**하는 것이다.

**네 단계**
1. 텍스트 → 토큰 → 토큰 ID → 임베딩
2. 간단한 Q·K·V와 Self-Attention 관찰
3. **Encoder–Decoder**로 영어→스페인어 번역
4. **Decoder-only**로 다음 토큰 예측, 텍스트 이어쓰기·질의응답

전체 실습에는 **훈련·파인튜닝·역전파 코드가 없다.** 모두 실행 가능한 완성형 코드이며, 학생은 문장과 설정을 바꿔 결과를 비교한다. **의미 검색과 RAG의 실제 구현은 P4 Project에서 수행한다.**


## 0. 실행 환경

- VS Code/Jupyter Notebook 또는 Google Colab에서 실행할 수 있다.
- 처음 실행하면 Hugging Face에서 **두 개의 사전훈련 모델**을 다운로드한다. 네트워크 속도에 따라 시간이 걸린다.
- PyTorch가 없는 환경이라면 별도로 `%pip install torch`를 실행한다. CPU만으로도 가능하지만 GPU보다 느릴 수 있다.
- 설치 직후 import 오류가 발생하면 **커널을 재시작한 후 아래 셀부터** 실행한다.

**사용 모델:** `Helsinki-NLP/opus-mt-en-es`(영어→스페인어, Encoder–Decoder 계열), `HuggingFaceTB/SmolLM2-135M-Instruct`(영어 중심의 소형 Decoder-only 모델). 최신 대형 대화형 모델 수준의 정확성을 기대하지 않는다.


In [ ]:
%pip install -q "transformers>=4.47,<5" sentencepiece sacremoses


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

# P4에서는 학습하지 않고 사전훈련 모델로 추론만 한다.
print("PyTorch:", torch.__version__)


---
## 1. 텍스트를 토큰·ID·256차원 벡터로 바꾸기

**목표:** 한 문장이 어떻게 `토큰 → 정수 ID → 256차원 임베딩`으로 변하는지 **모양과 그림으로 확인**한다. 직접 학습하지 않으며, 두 종류의 데이터를 명확히 구분한다.

- **교육용 숫자 예제(다운로드 불필요):** 우리가 정한 작은 어휘집과 임의 초기화된 `(어휘 수, 256)` 임베딩 표로 자료형·모양을 확인한다. **의미가 학습된 벡터가 아니다.**
- **실제 사전훈련 모델:** 뒤에서 사용할 영어→스페인어 번역 모델의 진짜 토크나이저와 학습된 임베딩을 관찰한다. **임베딩 차원은 해당 모델이 정한 값**이며 256이라고 가정하지 않는다.


### 1-1. 교육용 어휘집: 단어는 번호가 된다

`The red car is fast.`를 단순히 공백 기준으로 나눈다고 가정한다. 이는 **실제 Hugging Face 토크나이저와는 다른 설명용 처리**다.

어휘집에서 `the`에 2번, `red`에 3번 등의 ID를 부여하면 `ID`는 토큰의 위치를 가리키는 주소가 된다. ID의 숫자 차이는 의미의 거리가 아니다.


In [ ]:
# 수업용 단어 토큰화: 실제 모델의 subword 토큰화와 구별한다.
toy_text = "The red car is fast."
toy_tokens = toy_text.lower().replace(".", "").split()

# 작은 설명용 어휘집: 0=PAD, 1=UNK(미등록 단어)
toy_vocab = {"[PAD]": 0, "[UNK]": 1,
             "the": 2, "red": 3, "car": 4, "is": 5, "fast": 6,
             "and": 7, "blue": 8}
toy_ids = np.array([toy_vocab.get(token, 1) for token in toy_tokens])

print("원문:", toy_text)
print("토큰:", toy_tokens)
print("토큰 ID:", toy_ids.tolist())
print("미등록 단어 'swift'의 ID:", toy_vocab.get("swift", 1))


### 1-2. 토큰 ID로 256차원 벡터 조회하기

**임베딩 테이블**은 `어휘 수 × 임베딩 차원` 크기의 숫자 표다. 아래에서는 예제용 숫자를 임의로 만들고, 토큰 ID에 해당하는 행을 조회한다.

실제 사전훈련 모델에서는 이 표의 값이 훈련을 거쳐 결정되지만, **아래 숫자는 무작위로 생성되어 의미를 나타내지 않는다.** 관찰할 것은 **표의 구조와 256차원이라는 모양**이다.


In [ ]:
# 256차원 "설명용" 임베딩: 무작위 값이며 학습된 단어 의미를 담지 않는다.
EMBED_DIM = 256
rng_emb = np.random.default_rng(42)
toy_embedding_table = rng_emb.normal(0, 0.08, size=(len(toy_vocab), EMBED_DIM))
toy_vectors = toy_embedding_table[toy_ids]

print("임베딩 테이블 모양 (어휘 수, 차원):", toy_embedding_table.shape)
print("입력 토큰 ID 모양:", toy_ids.shape)
print("문장 임베딩 모양 (토큰 수, 차원):", toy_vectors.shape)
print("'car' 256차원 벡터 중 앞 8개:", np.round(toy_vectors[2, :8], 3))

# 같은 ID가 문장에 두 번 나타나면 두 위치에서 같은 행을 조회한다.
repeat_ids = np.array([toy_vocab['the'], toy_vocab['car'], toy_vocab['the']])
repeat_vectors = toy_embedding_table[repeat_ids]
print("같은 'the' 토큰의 두 임베딩이 같은가?:",
      np.allclose(repeat_vectors[0], repeat_vectors[2]))


In [ ]:
# 세로: 토큰 5개, 가로: 임베딩 차원 256개. 256개 값을 계산하지 않고 색으로 관찰한다.
fig, ax = plt.subplots(figsize=(11, 2.8))
im = ax.imshow(toy_vectors, aspect="auto", cmap="coolwarm", vmin=-0.25, vmax=0.25)
ax.set_yticks(range(len(toy_tokens)), toy_tokens)
ax.set_xticks([0, 31, 63, 95, 127, 159, 191, 223, 255],
              ["1", "32", "64", "96", "128", "160", "192", "224", "256"])
ax.set_xlabel("Embedding dimension (256 numeric values)")
ax.set_title("Illustrative 256-D embeddings (random, NOT trained)")
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
plt.show()


**그림 읽기:** 하나의 가로줄이 **토큰 하나의 256차원 벡터**다. 가로축의 256은 '단어 256개'가 아니라 **벡터에 포함된 숫자 256개**를 뜻한다. 색이 비슷하다고 해서 의미가 비슷한 단어라고 결론 내릴 수 없다(무작위 예제이기 때문).

### 1-3. 위치 정보

`The dog chased the boy.`와 `The boy chased the dog.`처럼 단어가 바뀌면 문장의 의미도 달라진다. 원래 Transformer는 **토큰 임베딩에 위치 인코딩을 더해** 순서를 나타낸다. 

여기서는 위치 벡터의 **모양이 `(5, 256)`이며 토큰 임베딩과 더해도 `(5, 256)`으로 유지된다**는 점만 확인하자.


In [ ]:
# 설명용 위치 벡터: 이 예제에서는 위치마다 서로 다른 숫자를 부여한다.
# 원래 Transformer는 사인/코사인 위치 인코딩을 사용했다. 여기서는 "모양"만 관찰한다.
rng_pos = np.random.default_rng(7)
toy_positions = rng_pos.normal(0, 0.04, size=toy_vectors.shape)
toy_input_vectors = toy_vectors + toy_positions

print("토큰 임베딩:", toy_vectors.shape)
print("위치 정보:", toy_positions.shape)
print("합친 입력 표현:", toy_input_vectors.shape)
print("같은 256차원이지만 위치 정보가 더해져 값이 달라짐:",
      not np.allclose(toy_input_vectors[0], toy_vectors[0]))


### 1-4. 이제 실제 사전훈련 모델에서 확인

다음 셀은 영어→스페인어 번역 모델 **MarianMT**의 실제 토크나이저를 사용한다. 토큰 수가 교육용 단어 분할과 다를 수 있다. 이 단계부터는 인터넷 다운로드가 필요하다.


In [ ]:
# 번역 모델의 토크나이저만 먼저 다운로드
MT_ID = "Helsinki-NLP/opus-mt-en-es"
mt_tokenizer = AutoTokenizer.from_pretrained(MT_ID)

sentence = "The red car is fast."
encoded = mt_tokenizer(sentence, return_tensors="pt")

print("원문:", sentence)
print("토큰:", mt_tokenizer.tokenize(sentence))
print("토큰 ID:", encoded["input_ids"][0].tolist())
print("다시 복원:", mt_tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True))


**관찰 포인트:** 출력의 토큰 수가 반드시 다섯 개는 아니다. `</s>` 등 특수 토큰이 추가될 수도 있다. 토큰 ID는 단순한 정수 번호다.

### 실제 사전훈련 임베딩의 차원

아래 셀은 실제 번역 모델의 Encoder 입력용 임베딩 벡터를 조회한다. **교육용 256차원**과 **실제 모델의 임베딩 차원**이 다를 수 있으므로, 출력된 `shape`를 그대로 해석한다.


In [ ]:
# 사전훈련된 번역 모델을 한 번만 다운로드한다. 3절에서도 같은 모델을 재사용한다.
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_ID)
mt_model.eval()  # 추론 모드: 가중치를 훈련하지 않음

with torch.inference_mode():
    word_vectors = mt_model.get_encoder().embed_tokens(encoded["input_ids"])

print("토큰 ID 배열 모양:", tuple(encoded["input_ids"].shape))
print("임베딩 배열 모양:", tuple(word_vectors.shape))
print("첫 토큰 벡터의 앞부분:", word_vectors[0, 0, :6].tolist())


**해석:** `(1, 토큰 수, 임베딩 차원)`에서 첫 번째 `1`은 배치의 문장이 하나라는 뜻이다. 256차원 예제는 구조 이해를 위한 것이고, 실제 모델의 차원은 구조마다 달라진다. 여기서는 **학습된 토큰 임베딩의 값**만 조회했다. 위치 정보까지 포함한 최종 Encoder 입력/출력을 모두 보여준 것은 아니다.

**정리 질문:** 임베딩 테이블 전체의 모양과 한 문장의 임베딩 모양은 무엇이 다른가? 숫자 ID가 비슷하면 단어 뜻도 비슷한가?


---
## 2. Self-Attention과 Q·K·V의 원리 관찰

이제 토큰을 실수 벡터로 나타낼 수 있다. 그런데 문장에서 `chased`를 표현할 때 `dog`와 `ball`의 정보를 참고하려면 어떻게 해야 할까?

**Q(Query)**는 현재 토큰이 찾으려는 정보, **K(Key)**는 각 토큰의 비교용 표지, **V(Value)**는 실제로 반영할 정보에 해당한다.

Self-Attention의 기본 흐름은 **① Q·K 관련성 계산 → ② 가중치 정규화(Softmax) → ③ V의 가중합**이다. 두 단계로 관찰한다.

- 먼저 **2차원짜리 장난감 숫자**로 연산을 이해한다.
- 이어서 **256차원짜리 다섯 토큰**으로 전체 가중치 히트맵을 그린다. **256개의 숫자를 직접 계산할 필요는 없다.**

> 모든 예시의 Q·K·V 값은 원리 설명을 위해 만든 것으로, 사전훈련된 번역 모델의 실제 Attention이 아니다.


### 2-1. 아주 작은 예제: Q·K가 가중치를 정한다

`the dog runs`에서 `runs`가 어떤 토큰을 참고할지 계산한다고 생각하자. 

- Q와 K를 비교해 점수를 만든다.
- **Softmax**로 점수를 더하면 1이 되는 가중치로 바꾼다.
- V에 가중치를 곱해 새로운 표현을 구한다.

다음 코드는 *전체 과정을 보여주기 위한 2차원 예제*다.


In [ ]:
# 예제: 'runs'가 다른 토큰 중 어떤 정보를 많이 참고할지 계산
words = ["the", "dog", "runs"]

query = np.array([1.0, 0.0])            # 현재 토큰의 Q (임의 설정)
keys = np.array([                       # 각 토큰의 K (임의 설정)
    [0.0, 1.0],  # the
    [2.0, 0.0],  # dog
    [0.5, 0.5],  # runs
])
values = np.array([                     # 각 토큰의 V (임의 설정)
    [1.0, 0.0],
    [0.0, 2.0],
    [1.0, 1.0],
])

scores = keys @ query / np.sqrt(2)      # Q와 K의 관련성 점수
weights = np.exp(scores - scores.max())
weights = weights / weights.sum()       # 합이 1이 되도록 변환
context = weights @ values             # V의 가중평균

for word, weight in zip(words, weights):
    print(f"{word:>5}: attention weight = {weight:.3f}")
print("가중치의 합:", round(float(weights.sum()), 3))
print("새로운 문맥 벡터:", context.round(3))


In [ ]:
# Attention 가중치를 막대그래프로 보기
plt.figure(figsize=(6, 3))
plt.bar(words, weights)
plt.title("Toy attention weights (not learned)")
plt.ylabel("Attention weight")
plt.ylim(0, 1)
plt.show()


**읽는 법:** 막대가 높을수록 해당 토큰의 **V 벡터**를 더 크게 반영한다. `weights`의 합이 `1`인 것을 확인해 보자. 실제로는 Q·K·V가 입력 임베딩에 학습된 선형 변환을 적용해 만들어진다.

### 2-2. 다섯 토큰을 256차원으로 표현하면?

문장 `The dog chased the ball.`의 다섯 토큰을 생각해 보자. 다음 코드에서는 `(5, 256)` 크기의 교육용 입력 행렬을 만든 뒤, **세 가지 서로 다른 변환**으로 Q·K·V를 만든다.

이 변환은 설명을 위해 **우리가 미리 설정한 값**이므로, `chased`가 `dog`와 `ball`을 상대적으로 더 참고하도록 설계되어 있다. **이런 관계가 저절로 학습되었다는 의미가 아니다.**


In [ ]:
# 다섯 토큰 × 256차원: 숫자를 일일이 계산하지 않고 모양을 확인한다.
attn_tokens = ["the", "dog", "chased", "the", "ball"]
d_model = 256
X = np.zeros((len(attn_tokens), d_model), dtype=np.float32)

# 토큰의 몇 개 차원에만 예시용 특징을 부여 (의미를 학습한 값이 아님)
X[1, 1] = 1.0  # dog
X[2, 3] = 1.0  # chased
X[4, 2] = 1.0  # ball

# 예시용 Q/K/V 변환. 실제 모델에서는 이 행렬들이 사전훈련을 거친다.
W_Q = np.zeros((d_model, d_model), dtype=np.float32)
W_K = np.zeros((d_model, d_model), dtype=np.float32)
W_V = np.eye(d_model, dtype=np.float32)
W_Q[3, 1] = 32.0  # chased의 Q가 dog를 참고하도록 설계
W_Q[3, 2] = 32.0  # chased의 Q가 ball을 참고하도록 설계
W_Q[1, 1] = 16.0
W_Q[2, 2] = 16.0
W_K[1, 1] = 1.0
W_K[2, 2] = 1.0

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("입력 임베딩 X:", X.shape)
print("Q, K, V:", Q.shape, K.shape, V.shape)


### 2-3. Attention 점수 → Softmax → V의 가중합

가로와 세로에 각각 다섯 토큰을 놓으면 **5×5 Attention 가중치 표**를 만들 수 있다. 각 행의 값은 합이 1이다. 그 다음 가중치를 **V(5×256)**에 적용하여 **출력(5×256)**을 만든다.


In [ ]:
# 점수: Q와 K의 유사도. 분모는 차원에 따른 점수 크기를 조정한다.
score_matrix = (Q @ K.T) / np.sqrt(d_model)  # (5, 5)

# 각 행마다 Softmax: 합이 1인 Attention 가중치
z = np.exp(score_matrix - score_matrix.max(axis=1, keepdims=True))
attention_weights = z / z.sum(axis=1, keepdims=True)

# Value의 가중합: 문맥이 반영된 토큰 표현
context_vectors = attention_weights @ V

print("Attention 점수 모양:", score_matrix.shape)
print("Attention 가중치 모양:", attention_weights.shape)
print("출력 문맥 벡터 모양:", context_vectors.shape)
print("'chased'가 참고한 토큰별 가중치:")
for token, value in zip(attn_tokens, attention_weights[2]):
    print(f"  {token:>7}: {value:.3f}")
print("'chased' 가중치 합:", round(float(attention_weights[2].sum()), 3))


In [ ]:
# 5×5 히트맵: 행은 현재 토큰(Query), 열은 참고하는 토큰(Key)
attn_labels = ["the [0]", "dog", "chased", "the [3]", "ball"]
fig, ax = plt.subplots(figsize=(6.7, 4.7))
img = ax.imshow(attention_weights, cmap="Blues", vmin=0, vmax=0.5)
ax.set_xticks(range(5), attn_labels, rotation=30, ha="right")
ax.set_yticks(range(5), attn_labels)
ax.set_xlabel("Key: token being referenced")
ax.set_ylabel("Query: current token")
ax.set_title("Illustrative Self-Attention (NOT pretrained)")
for i in range(5):
    for j in range(5):
        val = attention_weights[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val > 0.30 else "black", fontsize=9)
fig.colorbar(img, ax=ax, fraction=0.045, pad=0.03)
plt.show()


**히트맵 읽기:** 세 번째 행(`chased`)에서 `dog`와 `ball`의 칸이 상대적으로 진하게 나타난다. **우리가 그렇게 설계한 교육용 예시**이기 때문이다. 다른 토큰의 가중치도 각 행에서 확인할 수 있다.

중요한 점은 **5개 토큰의 Attention 가중치 표가 `(5, 5)`이지만, 출력 벡터는 여전히 `(5, 256)`**이라는 것이다. Attention은 토큰의 차원을 5로 줄이는 계산이 아니라 **다른 토큰들의 정보를 섞어 각 토큰의 표현을 갱신하는 계산**이다.

### 2-4. 원래 Transformer의 Encoder와 Decoder에서 사용 방식

| 구조 | 어떤 토큰을 참고하는가? |
|---|---|
| **Encoder Self-Attention** | 입력 문장의 앞뒤 토큰 전체 |
| **Decoder Masked Self-Attention** | 이전에 생성한 토큰과 현재 위치까지만 (미래 출력은 차단) |
| **Decoder Cross-Attention** | 현재 Decoder 표현을 Query로 삼고 Encoder 출력에서 Key·Value를 가져옴 |

**Multi-Head Attention**은 이런 계산을 여러 Head에서 따로 수행해 결합한다. 예를 들어 256차원 표현을 4개 Head가 각각 64차원으로 살펴보도록 설계할 수 있지만, **Head마다 반드시 64차원이어야 하는 것은 아니다.**

**확인 질문:** `Q·K`가 직접 `Value`를 나타내는가? `V`는 어느 단계에서 사용되는가? 가중치가 높다고 실제 모델이 문법적 관계를 정확히 판별했다고 결론 내릴 수 있는가?


---
## 3. Encoder–Decoder 활용: 영어 → 스페인어 번역

**번역 모델을 직접 훈련하지 않는다.** 사전훈련된 Encoder–Decoder 계열의 Transformer를 가져와 새 영어 문장들을 번역한다.

원리: **영어 입력 → Encoder 문맥 표현 → Decoder가 다음 스페인어 토큰을 반복 생성**

이전에 불러온 `mt_tokenizer`와 `mt_model`을 재사용한다.


In [ ]:
# 수업용 예문: 기존 이론 노트북의 영어→스페인어 예시를 일부 활용
english_sentences = [
    "How is the weather today?",
    "You are welcome.",
    "The red car is fast.",
    "I like reading books.",
]

inputs = mt_tokenizer(
    english_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
)

# 모델 가중치를 바꾸지 않고 번역 결과만 생성
with torch.inference_mode():
    output_ids = mt_model.generate(**inputs, max_new_tokens=60)

spanish_sentences = mt_tokenizer.batch_decode(
    output_ids, skip_special_tokens=True
)

for en, es in zip(english_sentences, spanish_sentences):
    print("EN:", en)
    print("ES:", es)
    print("-" * 44)


### 관찰 및 해석

1. `The red car`의 `red`와 `car`의 위치가 스페인어에서 그대로 유지되는가?
2. `You are welcome.`의 번역은 하나만 가능한가?
3. 번역 모델이 영어 단어와 스페인어 단어를 **같은 위치끼리 대응**시키는 방식이라고 생각하는가?

**추가 실험:** `english_sentences` 목록의 문장을 한두 개 바꾼 뒤 같은 셀을 다시 실행해 보자. 훈련이나 설정 변경은 필요하지 않다.


### 세 가지 Transformer 구조의 현재 활용

| 구조 | 대표 활용 | 이 실습에서 |
|---|---|---|
| **Encoder-only** (BERT 등) | 텍스트 분류·의미 검색·임베딩 | **개념만 소개** — 모델 실행 없음 |
| **Encoder–Decoder** (MarianMT 등) | 번역·요약·음성 인식 | **영어→스페인어 번역** |
| **Decoder-only** (GPT 계열 등) | 글쓰기·질의응답·코드 생성 | **이어쓰기·질의응답** |

예를 들어 Encoder-only 기반 임베딩 모델은 `반려견 관리`와 `강아지 돌보기`처럼 표현이 달라도 의미가 비슷한 문서를 찾는 데 활용할 수 있다. **이 구분은 대표적 용도일 뿐**, Decoder-only 모델도 번역 등을 수행할 수 있다.



### 번역 모델에서 LLM으로 연결하기

방금 사용한 모델은 **Encoder–Decoder Transformer**다. 영어 문장을 Encoder가 처리하고, Decoder는 그 내용을 참고하여 스페인어 번역을 생성했다.

이제 사용할 모델은 **Decoder-only Transformer 기반의 소형 언어 모델**이다. 별도의 Encoder나 Cross-Attention 없이 **프롬프트와 이미 생성한 토큰을 한 줄의 시퀀스로 처리**하며, 다음 토큰을 반복 예측한다.

| 이번 Lab의 모델 | 모델 구조 | 무엇을 확인할까? |
|---|---|---|
| 영어→스페인어 번역 모델 | Encoder–Decoder | 입력 문장을 참고한 번역문 생성 |
| 소형 지시문 응답 언어 모델 | Decoder-only | 프롬프트를 이어 쓰고 질문에 답하는 과정 |

두 모델 모두 **이미 학습된 가중치를 불러와 사용**한다. 여기서 *LLM*이라는 말은 단순히 Decoder라는 블록 자체를 뜻하지 않고, **많은 데이터로 학습된 언어 모델**을 가리킨다. 이번 실습의 소형 모델은 이 원리를 관찰하기 위한 교육용 예제이며, 대형 상용 LLM의 성능을 대표하지 않는다.


---
## 4. Decoder-only: 다음 토큰을 예측하고 문장을 생성하기

이번에는 Encoder가 따로 없는 **Decoder-only** 모델을 사용한다. 입력 프롬프트를 보고 다음 토큰의 확률을 계산한 다음, 토큰을 하나씩 추가한다.

빠른 실습을 위해 **작은 모델**을 사용한다. 따라서 출력이 어색하거나 틀릴 수 있다. 여기서 보는 것은 답변 품질이 아니라 **어떻게 텍스트를 생성하는지**이다.


In [ ]:
LM_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_ID)
lm_model = AutoModelForCausalLM.from_pretrained(LM_ID)
lm_model.eval()

print("사용하는 Decoder-only 모델:", LM_ID)


### 4-1. 다음 토큰 후보의 확률 확인

모델에 문장을 넣고 **마지막 위치**에서 다음 토큰의 확률을 확인한다. 예제에서는 가장 가능성이 높다고 예측한 상위 5개 토큰을 살펴본다.

실제 모델의 확률이므로 교재의 가상 예시와 일치하지 않을 수 있다.


In [ ]:
prompt = "The capital of France is"
encoded_lm = lm_tokenizer(prompt, return_tensors="pt")

with torch.inference_mode():
    logits = lm_model(**encoded_lm).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)

values, indices = torch.topk(probs, k=5)
print("입력:", prompt)
print("다음 토큰 후보:")
for token_id, prob in zip(indices.tolist(), values.tolist()):
    token = lm_tokenizer.decode([token_id])
    print(f"  {token!r:<18} 확률: {prob:.3f}")


### 4-2. 활용 ① 텍스트 이어쓰기 / ② 간단한 질의응답

모델에는 텍스트를 이어 쓰라는 **지시문** 또는 질문을 넣을 수 있다. 두 작업 모두 내부적으로는 다음 토큰 생성이 반복된다.

아래 함수는 '질문/지시문 → 생성 결과'를 간단히 확인하기 위한 것이다. **Chat Template**은 역할(user/assistant)을 모델이 이해하는 입력 형식으로 바꿔준다. 내부 상세 문법은 알 필요가 없다.


In [ ]:
def ask_model(user_message, max_new_tokens=60):
    # 학생은 질문만 바꾼다. 아래 부분은 사전훈련 모델 추론용 공통 코드다.
    messages = [{"role": "user", "content": user_message}]
    token_ids = lm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )

    with torch.inference_mode():
        generated = lm_model.generate(
            token_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # 매번 가장 확률이 높은 토큰 선택 (Greedy)
            pad_token_id=lm_tokenizer.eos_token_id,
        )

    # 모델에 입력한 부분을 제외하고 새로 생성한 부분만 보여준다.
    new_tokens = generated[0, token_ids.shape[-1]:]
    return lm_tokenizer.decode(new_tokens, skip_special_tokens=True)

# 활용 ① 텍스트 이어쓰기
story_prompt = "Continue in one simple sentence: A little dog found a red ball."
print("[이어쓰기]")
print(ask_model(story_prompt, max_new_tokens=45))

# 활용 ② 간단한 질의응답
question = "What is the capital of Spain? Answer in one short sentence."
print()
print("[질의응답]")
print(ask_model(question, max_new_tokens=40))


**예제에서 확인할 점:** 이 모델이 답을 맞히거나 말을 자연스럽게 한다는 사실보다, 두 작업을 **같은 Decoder-only 모델**로 수행했다는 사실이 핵심이다. 아주 작은 교육용 모델이므로 정답을 틀리거나 의미 없는 문장을 생성할 수 있다.


---
## 5. P4 Project 미리보기 — 의미 검색과 답변 생성 연결하기

이번 Lab에서는 **번역(Encoder–Decoder)**과 **텍스트 생성(Decoder-only)**을 각각 실행했다. Project에서는 아직 실행하지 않은 **Encoder-only 계열의 문장 임베딩 모델**을 추가하여 두 기능을 연결한다.

**프로젝트 질문 예시:** `같은 동물을 여러 번 촬영했을 때 개체 수를 어떻게 세나요?`

| 단계 | 사용하는 모델 또는 자료 | 학생이 관찰할 내용 |
|---|---|---|
| ① 문서 준비 | 가상 한솔 생태관측소 안내문 **25개** | 문서마다 서로 다른 정보를 담고 있음 |
| ② 의미 검색 | **Encoder 계열** 문장 임베딩 모델 | 단어가 정확히 같지 않아도 관련 문서 검색 |
| ③ 답변 생성 | **Decoder-only** 언어 모델 | 검색된 문서를 읽고 질문에 대한 문장 생성 |
| ④ 결과 검토 | 학생이 원문과 비교 | 답변에 근거가 있는지, 잘못 만든 내용이 있는지 판단 |

```text
질문 → 의미가 유사한 문서 3개 찾기 → 문서를 함께 전달 → 답변 생성 → 근거 확인
       (Encoder 계열)                 (Decoder-only)
```

**Lab과 Project의 차이:** 이 Lab에서는 이미 준비된 모델의 입력과 출력을 *관찰*했다. Project에서는 **단순 키워드 검색과 의미 검색을 비교하고**, 검색 결과를 생성 모델에 제공하여 **답변의 근거 적합성**까지 평가한다.

**필요한 파일:** `P4_project_document_search_RAG.ipynb`, `P4_project_documents_25.json` (같은 폴더에 보관). Project에서도 **모델 훈련은 없다.**

## 6. 실습 마무리 — 결과를 근거로 설명하기

다음 질문에 코드 출력을 보고 **각각 1~2문장**으로 답해 보자.

1. 토큰과 토큰 ID, 임베딩 벡터는 무엇이 다른가?
2. Attention 예제에서 Q·K와 V의 역할은 무엇이었나?
3. Encoder–Decoder 번역에서는 Encoder와 Decoder가 각각 무엇을 했는가?
4. Decoder-only 모델은 왜 영어 프롬프트를 입력받아도 Encoder가 필요 없는가?
5. 번역과 질의응답 모두 **다음 토큰 생성**이라는 공통 원리를 공유한다는 설명은 어떤 뜻인가?
6. Decoder-only **구조**와 이를 사전훈련한 **언어 모델**을 어떻게 구분할 수 있을까?
7. Project에서는 Encoder 계열과 Decoder-only 모델이 각각 어떤 역할을 하는가?

**범위 확인:** 이 노트북에는 학습 루프·손실함수 최적화·파인튜닝·강화학습 코드가 없다. 전체 코드는 **사전훈련된 모델의 추론 또는 숫자 예제**이다.

### 모델 출처

- [Helsinki-NLP/opus-mt-en-es](https://huggingface.co/Helsinki-NLP/opus-mt-en-es)
- [HuggingFaceTB/SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)

**다음 학습:** `P4_project_document_search_RAG.ipynb` — 키워드 검색·의미 검색·근거 기반 답변 생성 비교.
